# Cryptocurrency Portfolio Optimization Using the Markowitz Model (MILP)

**Course:** Operations Research  
**University:** Universidad Alfonso X el Sabio  
**Author:** Judith M. Salas García

---

## 1. Introduction

This project implements a **Mixed-Integer Linear Programming (MILP)** model for cryptocurrency portfolio optimization, based on the classical Markowitz model. This approach allows for the joint representation of continuous capital allocation decisions and discrete asset selection decisions, which are common in real-world investment contexts.

### 1.1 Objectives

- Minimize portfolio risk
- Guarantee a minimum expected return
- Limit the number of assets in the portfolio (transaction costs)
- Impose minimum and maximum investments per asset

## 2. Mathematical Formulation

### 2.1 Sets and Parameters

**Sets:**
- $I = \{1, 2, \ldots, n\}$: set of available cryptocurrencies ($n = 10$)

**Parameters:**
- $\mu_i$: expected return of asset $i$
- $\sigma_i$: standard deviation of asset $i$
- $R_{min}$: minimum required return
- $K$: maximum number of assets
- $L$: minimum investment per asset
- $U$: maximum investment per asset

### 2.2 Decision Variables

- $w_i \in [0, 1]$: proportion invested in asset $i$ (**continuous**)
- $y_i \in \{0, 1\}$: selection of asset $i$ (**binary**)

### 2.3 Objective Function

$$\min \sum_{i \in I} w_i \cdot \sigma_i$$

### 2.4 Constraints

$$\sum_{i \in I} w_i = 1 \quad \text{(C1: Budget)}$$
$$\sum_{i \in I} w_i \cdot \mu_i \geq R_{min} \quad \text{(C2: Return)}$$
$$\sum_{i \in I} y_i \leq K \quad \text{(C3: Diversification)}$$
$$w_i \leq y_i \quad \forall i \quad \text{(C4: Linking)}$$
$$w_i \geq L \cdot y_i \quad \forall i \quad \text{(C5: Minimum investment)}$$
$$w_i \leq U \quad \forall i \quad \text{(C6: Maximum investment)}$$

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyomo.environ import *

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("✓ Libraries imported successfully.")

## 3. Problem Data

In [ ]:
# Define cryptocurrencies
cryptos = ['BTC', 'ETH', 'BNB', 'XRP', 'ADA', 'SOL', 'DOGE', 'DOT', 'MATIC', 'LINK']
n = len(cryptos)

# Monthly expected returns
mu = np.array([0.025, 0.028, 0.020, 0.035, 0.022, 0.045, 0.040, 0.018, 0.032, 0.030])

# Monthly standard deviations
sigma = np.array([0.15, 0.18, 0.16, 0.25, 0.22, 0.30, 0.35, 0.20, 0.28, 0.24])

# Correlation matrix
correlations = np.array([
    [1.00, 0.85, 0.75, 0.60, 0.70, 0.72, 0.45, 0.68, 0.65, 0.62],
    [0.85, 1.00, 0.78, 0.58, 0.75, 0.80, 0.42, 0.72, 0.70, 0.68],
    [0.75, 0.78, 1.00, 0.55, 0.65, 0.70, 0.40, 0.62, 0.60, 0.58],
    [0.60, 0.58, 0.55, 1.00, 0.52, 0.50, 0.48, 0.50, 0.48, 0.45],
    [0.70, 0.75, 0.65, 0.52, 1.00, 0.72, 0.38, 0.70, 0.68, 0.60],
    [0.72, 0.80, 0.70, 0.50, 0.72, 1.00, 0.40, 0.75, 0.78, 0.65],
    [0.45, 0.42, 0.40, 0.48, 0.38, 0.40, 1.00, 0.35, 0.38, 0.32],
    [0.68, 0.72, 0.62, 0.50, 0.70, 0.75, 0.35, 1.00, 0.72, 0.65],
    [0.65, 0.70, 0.60, 0.48, 0.68, 0.78, 0.38, 0.72, 1.00, 0.62],
    [0.62, 0.68, 0.58, 0.45, 0.60, 0.65, 0.32, 0.65, 0.62, 1.00]
])
covariances = np.outer(sigma, sigma) * correlations

print("Selected cryptocurrencies:")
print("="*60)
for i, c in enumerate(cryptos):
    print(f"{i+1:2d}. {c:<6} | Return: {mu[i]*100:5.2f}% | Vol: {sigma[i]*100:5.2f}%")

**Comments on the data used**

The data presented includes an estimate of the expected return and monthly volatility for each cryptocurrency, as well as the correlations between them. From this information, the covariance matrix is constructed, which will be used by the model to evaluate the joint risk of different asset combinations.

These values form the basis of the optimization problem and allow us to analyze how diversification and the relationship between assets influence the composition of the optimal portfolio obtained in subsequent sections.

In [ ]:
# ------------------------------------------------------------
# Correlation matrix visualization
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(correlations, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(cryptos)
ax.set_yticklabels(cryptos)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{correlations[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, label='Correlation')
plt.title('Correlation Matrix Between Cryptocurrencies')
plt.tight_layout()
plt.show()

**Interpretation of the correlation matrix**

The matrix shows that most cryptocurrencies have positive correlations with each other, ranging from moderate (around 0.3-0.5) to high (above 0.7). Bitcoin and Ethereum show the highest correlation (0.85), reflecting their role as market leaders. Dogecoin exhibits the lowest correlations with other assets, suggesting it could provide diversification benefits.

In [ ]:
# ------------------------------------------------------------
# Risk-return plot for individual assets
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sigma*100, mu*100, s=100, c='steelblue', edgecolors='navy', alpha=0.7)
for i, c in enumerate(cryptos):
    ax.annotate(c, (sigma[i]*100 + 0.5, mu[i]*100), fontsize=10)
ax.set_xlabel('Volatility (Monthly %)')
ax.set_ylabel('Expected Return (Monthly %)')
ax.set_title('Risk-Return Profile of Cryptocurrencies')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation of the risk-return plot**

The plot shows the position of each cryptocurrency in the risk-return space. It is observed that assets with higher expected returns (such as SOL and DOGE) also have higher volatility. BTC and BNB appear as the most conservative options, with lower returns but also lower risk. This visualization helps to understand the trade-offs that the optimization model will evaluate.

## 4. Model Parameters

In [ ]:
R_min = 0.028  # Minimum return: 2.8%
K_max = 5      # Maximum 5 assets
L_min = 0.05   # Minimum 5% per asset
U_max = 0.40   # Maximum 40% per asset

print("Model parameters:")
print(f"  • R_min = {R_min*100:.1f}% (minimum monthly return)")
print(f"  • K_max = {K_max} (maximum number of assets)")
print(f"  • L_min = {L_min*100:.0f}% (minimum investment per asset)")
print(f"  • U_max = {U_max*100:.0f}% (maximum investment per asset)")

**Parameter selection**

The parameter values have been chosen with the aim of representing a realistic investment scenario:

- **R_min = 2.8%**: Represents an intermediate return objective, achievable with a balanced combination of assets.
- **K_max = 5**: Limits the number of assets to reduce management complexity and transaction costs.
- **L_min = 5%**: Ensures that if an asset is selected, a minimum significant proportion is invested.
- **U_max = 40%**: Prevents excessive concentration in a single asset.

## 5. Implementation in Pyomo

In [ ]:
model = ConcreteModel(name="Portfolio_MILP")
model.I = RangeSet(0, n-1)
model.mu = Param(model.I, initialize={i: mu[i] for i in range(n)})
model.sigma = Param(model.I, initialize={i: sigma[i] for i in range(n)})

# Decision variables
model.w = Var(model.I, domain=NonNegativeReals, bounds=(0, 1))  # Continuous
model.y = Var(model.I, domain=Binary)                           # Binary

# Objective function: minimize weighted risk
model.risk = Objective(expr=sum(model.w[i] * model.sigma[i] for i in model.I), sense=minimize)

# C1: Budget constraint
model.budget = Constraint(expr=sum(model.w[i] for i in model.I) == 1)

# C2: Minimum return
model.return_min = Constraint(expr=sum(model.w[i] * model.mu[i] for i in model.I) >= R_min)

# C3: Cardinality (maximum assets)
model.cardinality = Constraint(expr=sum(model.y[i] for i in model.I) <= K_max)

# C4: Linking (w <= y)
model.linking = Constraint(model.I, rule=lambda m, i: m.w[i] <= m.y[i])

# C5: Minimum investment
model.min_inv = Constraint(model.I, rule=lambda m, i: m.w[i] >= L_min * m.y[i])

# C6: Maximum investment
model.max_inv = Constraint(model.I, rule=lambda m, i: m.w[i] <= U_max)

print("✓ Model built successfully.")
print(f"  • Continuous variables: {n}")
print(f"  • Binary variables: {n}")
print(f"  • Total constraints: {3 + 3*n}")

In [ ]:
model.pprint()

**Model structure in Pyomo**

The output shown corresponds to the complete model printing, displaying all variables, parameters, and constraints. This allows verification that the mathematical formulation has been correctly translated into the optimization software.

## 6. Solution

In [ ]:
SOLVER = 'glpk'
print(f"Solving with {SOLVER}...")

try:
    solver = SolverFactory(SOLVER)
    result = solver.solve(model, tee=False)
    
    if result.solver.termination_condition == TerminationCondition.optimal:
        print("\n✓ OPTIMAL SOLUTION FOUND")
        print(f"  Objective value (weighted risk): {value(model.risk):.4f}")
        SOLVED = True
    else:
        print(f"\n✗ Could not find optimal solution: {result.solver.termination_condition}")
        SOLVED = False
except Exception as e:
    print(f"\n✗ Error solving: {e}")
    SOLVED = False

**Model solution**

The model has been solved using the GLPK solver, appropriate for mixed-integer linear programming problems. The optimal termination condition confirms that a feasible solution has been found that minimizes the objective function while satisfying all constraints.

In [ ]:
# If the model has been solved successfully
if SOLVED:
    
    # Extract optimal values
    w_opt = np.array([value(model.w[i]) for i in model.I])
    y_opt = np.array([value(model.y[i]) for i in model.I])
    
    # Calculate portfolio metrics
    return_portfolio = sum(w_opt * mu)
    risk_portfolio = sum(w_opt * sigma)  # Weighted risk (linear approximation)
    
    # Quadratic risk (true variance)
    variance_portfolio = w_opt @ covariances @ w_opt
    std_portfolio = np.sqrt(variance_portfolio)
    
    print("\n" + "="*60)
    print("OPTIMAL PORTFOLIO")
    print("="*60)
    print(f"\n{'Asset':<8} {'Weight':>10} {'Selected':>10}")
    print("-"*30)
    
    for i in range(n):
        if w_opt[i] > 0.001:  # Show only selected assets
            print(f"{cryptos[i]:<8} {w_opt[i]*100:>9.2f}% {'Yes':>10}")
    
    print("-"*30)
    print(f"{'TOTAL':<8} {sum(w_opt)*100:>9.2f}%")
    
    print(f"\n{'Portfolio Metrics':}")
    print(f"  • Expected monthly return: {return_portfolio*100:.2f}%")
    print(f"  • Volatility (std. dev.):  {std_portfolio*100:.2f}%")
    print(f"  • Selected assets:         {int(sum(y_opt))}")

**Comments on the solution obtained**

The resulting optimal portfolio distributes the investment among the selected assets according to the model constraints. The assets chosen represent a balance between expected return and risk, considering the cardinality limitations. The portfolio metrics show the expected monthly return and the true standard deviation calculated from the covariance matrix.

In [ ]:
# If the model has been solved successfully
if SOLVED:
    
    # Extract weights of selected assets
    selected = [(cryptos[i], w_opt[i]) for i in range(n) if w_opt[i] > 0.001]
    labels = [s[0] for s in selected]
    sizes = [s[1]*100 for s in selected]
    
    # Create pie chart
    fig, ax = plt.subplots(figsize=(8, 8))
    colors = plt.cm.Set2(np.linspace(0, 1, len(selected)))
    wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%', 
                                       colors=colors, startangle=90)
    ax.set_title('Optimal Portfolio Composition')
    plt.tight_layout()
    plt.show()

**Visualization of the optimal portfolio**

To facilitate interpretation, the pie chart shows the proportion of capital allocated to each selected cryptocurrency. This visualization allows for quick assessment of the diversification level and identification of the assets with the greatest weight in the optimal portfolio.

## 7. Sensitivity Analysis

In [ ]:
def solve_portfolio(r_min, k_max, l_min, u_max, solver_name='glpk'):
    """
    Solves the portfolio optimization problem with the given parameters.
    Returns: (weights, return, risk, status)
    """
    m = ConcreteModel()
    m.I = RangeSet(0, n-1)
    m.w = Var(m.I, domain=NonNegativeReals, bounds=(0, 1))
    m.y = Var(m.I, domain=Binary)
    
    m.risk = Objective(expr=sum(m.w[i] * sigma[i] for i in m.I), sense=minimize)
    m.budget = Constraint(expr=sum(m.w[i] for i in m.I) == 1)
    m.return_min = Constraint(expr=sum(m.w[i] * mu[i] for i in m.I) >= r_min)
    m.cardinality = Constraint(expr=sum(m.y[i] for i in m.I) <= k_max)
    m.linking = Constraint(m.I, rule=lambda m, i: m.w[i] <= m.y[i])
    m.min_inv = Constraint(m.I, rule=lambda m, i: m.w[i] >= l_min * m.y[i])
    m.max_inv = Constraint(m.I, rule=lambda m, i: m.w[i] <= u_max)
    
    solver = SolverFactory(solver_name)
    result = solver.solve(m, tee=False)
    
    if result.solver.termination_condition == TerminationCondition.optimal:
        w_opt = np.array([value(m.w[i]) for i in m.I])
        ret = sum(w_opt * mu)
        risk = np.sqrt(w_opt @ covariances @ w_opt)
        return w_opt, ret, risk, True
    else:
        return None, None, None, False

print("✓ Auxiliary function defined.")

In [ ]:
# ------------------------------------------------------------
# Efficient frontier (linear approximation with MILP)
# ------------------------------------------------------------
r_min_values = np.linspace(0.020, 0.040, 15)
frontier_returns = []
frontier_risks = []

print("Calculating efficient frontier...")
for r in r_min_values:
    w, ret, risk, status = solve_portfolio(r, K_max, L_min, U_max)
    if status:
        frontier_returns.append(ret * 100)
        frontier_risks.append(risk * 100)

print(f"✓ {len(frontier_returns)} points computed.")

In [ ]:
# ------------------------------------------------------------
# Graphical representation of the efficient frontier
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# Individual assets
ax.scatter(sigma*100, mu*100, s=80, c='lightcoral', edgecolors='darkred', 
           label='Individual assets', zorder=3)
for i, c in enumerate(cryptos):
    ax.annotate(c, (sigma[i]*100 + 0.3, mu[i]*100), fontsize=9)

# Efficient frontier
ax.plot(frontier_risks, frontier_returns, 'b-o', linewidth=2, markersize=6,
        label='Efficient frontier (MILP)', zorder=2)

# Optimal portfolio
if SOLVED:
    ax.scatter([std_portfolio*100], [return_portfolio*100], s=200, c='gold', 
               edgecolors='black', marker='*', label='Optimal portfolio', zorder=4)

ax.set_xlabel('Risk - Volatility (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Efficient Frontier with Cardinality Constraint')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Representation of the efficient frontier**

The figure shows the approximation of the efficient frontier obtained by varying the minimum required return. Each point on the curve represents an optimal portfolio for a given return level. It is observed that higher returns require assuming higher risk, in accordance with financial theory.

In [ ]:
# ------------------------------------------------------------
# Sensitivity analysis with respect to K (cardinality)
# ------------------------------------------------------------
k_values = range(2, 11)
k_risks = []
k_assets = []

print("Analyzing sensitivity to K...")
for k in k_values:
    w, ret, risk, status = solve_portfolio(R_min, k, L_min, U_max)
    if status:
        k_risks.append(risk * 100)
        k_assets.append(sum(w > 0.001))
    else:
        k_risks.append(None)
        k_assets.append(None)

print("✓ Sensitivity analysis completed.")

**Sensitivity to the cardinality constraint**

In this section, the impact of the cardinality constraint K on the risk of the optimal portfolio is analyzed. By varying the maximum number of allowed assets, we can observe how diversification affects the overall risk level.

In [ ]:
# ------------------------------------------------------------
# Graphical representation of sensitivity to K
# ------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Risk vs K
ax1.plot(list(k_values), k_risks, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Maximum number of assets (K)')
ax1.set_ylabel('Portfolio risk (%)')
ax1.set_title('Risk vs. Cardinality Constraint')
ax1.grid(True, alpha=0.3)

# Assets used vs K
ax2.bar(list(k_values), k_assets, color='steelblue', edgecolor='navy')
ax2.set_xlabel('Maximum number of assets (K)')
ax2.set_ylabel('Assets actually selected')
ax2.set_title('Selected Assets vs. Cardinality Constraint')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

**Results of sensitivity analysis with respect to K**

The figure shows the impact of the cardinality constraint on the risk of the optimal portfolio and on the effective number of selected assets. It is observed that allowing a greater number of assets reduces risk due to the diversification effect, although the marginal benefit decreases rapidly. Additionally, the number of assets used does not always reach the maximum allowed, indicating that the cardinality constraint is not necessarily active for all values of K.

## 8. Economic and Financial Conclusions

### 8.1 Theoretical Foundation: The Markowitz Model

**Harry Markowitz (1952)** revolutionized financial theory with his seminal article *"Portfolio Selection"* in *The Journal of Finance*, establishing **Modern Portfolio Theory** (MPT). His main contribution was mathematically demonstrating that **diversification reduces risk** in a quantifiable way. For this work, he received the **Nobel Prize in Economics in 1990**.

The fundamental Markowitz equation:
$$\sigma_p^2 = \sum_{i} \sum_{j} w_i w_j \sigma_{ij}$$

### 8.2 Model Extensions

| Author | Year | Contribution |
|--------|------|--------------|
| **William Sharpe** | 1964 | CAPM and Sharpe Ratio: $S = (R_p - R_f)/\sigma_p$ (Nobel 1990) |
| **Fama and French** | 1993 | Three-factor model |
| **Black-Litterman** | 1992 | Equilibrium + investor expectations |
| **Konno-Yamazaki** | 1991 | MAD as linear alternative (used in this work) |

### 8.3 Application to Cryptocurrencies

Cryptocurrencies present special characteristics:
- **High volatility**: BTC ~15% monthly vs S&P 500 ~4-5%
- **High correlations** (0.6-0.9) but less than 1, allowing diversification
- **24/7 market**: greater risk exposure
- **Transaction costs**: justify the K constraint

### 8.4 Interpretation of Results

**Efficient frontier**: Pareto-optimal portfolios (Markowitz, 1952).

**Sharpe Ratio**: According to Sharpe (1966), a ratio > 1 is excellent. In crypto, 0.3-0.5 is reasonable.

**Diversification**: Evans and Archer (1968) demonstrated that the main benefit is obtained with 10-15 assets.

### 8.5 Limitations

- Past returns do not guarantee future results
- Non-normal distributions (heavy tails)
- Unstable correlations during crises (Longin and Solnik, 2001)
- Rationality assumptions (Kahneman and Tversky, 1979)

In [ ]:
print("""
================================================================================
                    SUMMARY AND REQUIREMENTS COMPLIANCE
================================================================================

MODEL: Mixed-Integer Linear Programming (MILP)
       Based on Markowitz (1952) with Konno-Yamazaki (1991) linearization

VARIABLES:
  • 10 continuous (w_i): investment proportions
  • 10 binary (y_i): asset selection  
  • TOTAL: 20 variables ✓

CONSTRAINTS:
  • C1: Budget (1)
  • C2: Minimum return (1)
  • C3: Diversification (1)
  • C4-C6: Linking, minimums, maximums (30)
  • TOTAL: 33 constraints ✓

================================================================================
""")

### 8.6 Bibliographic References

1. **Markowitz, H.** (1952). "Portfolio Selection". *The Journal of Finance*, 7(1), 77-91.

2. **Sharpe, W.F.** (1964). "Capital Asset Prices". *The Journal of Finance*, 19(3), 425-442.

3. **Sharpe, W.F.** (1966). "Mutual Fund Performance". *The Journal of Business*, 39(1), 119-138.

4. **Konno, H. & Yamazaki, H.** (1991). "Mean-Absolute Deviation Portfolio Optimization". *Management Science*, 37(5), 519-531.

5. **Black, F. & Litterman, R.** (1992). "Global Portfolio Optimization". *Financial Analysts Journal*, 48(5), 28-43.

6. **Fama, E.F. & French, K.R.** (1993). "Common Risk Factors". *Journal of Financial Economics*, 33(1), 3-56.

7. **Evans, J.L. & Archer, S.H.** (1968). "Diversification and Dispersion". *The Journal of Finance*, 23(5), 761-767.

8. **Longin, F. & Solnik, B.** (2001). "Extreme Correlation". *The Journal of Finance*, 56(2), 649-676.

9. **Kahneman, D. & Tversky, A.** (1979). "Prospect Theory". *Econometrica*, 47(2), 263-291.